# 🐍 Python — Exception Handling & Object-Oriented Programming

---

This notebook is a complete guide to two of the most important pillars of professional Python:

| Section | Topics |
|---|---|
| **Part 1** | Exception Handling — try / except / else / finally |
| **Part 2** | OOP — Classes & Objects |
| **Part 3** | OOP — Inheritance |
| **Part 4** | OOP — Polymorphism |
| **Part 5** | OOP — Encapsulation |
| **Part 6** | OOP — Abstraction |
| **Part 7** | Magic Methods & Operator Overloading |

> 💡 **How to use this notebook:** Read the explanation in each markdown cell, then run the code cell below it. Try modifying the code and re-running to build intuition.

---
# Part 1 — Exception Handling

## What is an Exception?

An **exception** is an event that disrupts the normal flow of a program.  
When Python hits a problem it cannot continue past, it **raises** an exception object.

If that exception is never *caught*, the program crashes with a traceback.

Common built-in exceptions:

| Exception | When it happens |
|---|---|
| `ZeroDivisionError` | Division by zero |
| `NameError` | Variable not defined |
| `TypeError` | Wrong type for an operation |
| `ValueError` | Right type, wrong value (e.g. `int('abc')`) |
| `FileNotFoundError` | File does not exist |
| `AttributeError` | Object has no such attribute |
| `IndexError` | List index out of range |

In [ ]:
# --- Seeing exceptions in the wild ---
# Uncomment each line one at a time to see the error it produces.

# 1/0                  # ZeroDivisionError
# print(undefined_var) # NameError
# int('hello')         # ValueError
# 'text' + 5           # TypeError
# [1, 2, 3][99]        # IndexError

## 1.1 — The `try / except` Block

Wrap risky code in a `try` block.  
If an exception is raised, Python jumps to the matching `except` block instead of crashing.

```
try:
    <risky code>
except SomeError:
    <handle the problem>
```

In [ ]:
# Basic try / except
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Oops! You cannot divide by zero.")

print("The program keeps running after the except block.")

In [ ]:
# Catching the exception object gives you the error message
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Caught a ZeroDivisionError: {e}")

## 1.2 — Multiple `except` Clauses

You can chain multiple `except` clauses to handle different error types differently.  
Python tries them top-to-bottom and runs only the **first** one that matches.  
Put the most specific exceptions first, and use `Exception` as a catch-all at the end.

In [ ]:
def safe_divide(numerator, denominator_str):
    """Convert a string to int and divide — handles multiple failure modes."""
    try:
        denominator = int(denominator_str)   # could raise ValueError
        result = numerator / denominator     # could raise ZeroDivisionError
        return result
    except ValueError:
        print(f"  ❌ '{denominator_str}' is not a valid number.")
    except ZeroDivisionError:
        print("  ❌ Cannot divide by zero.")
    except Exception as e:
        print(f"  ❌ Unexpected error: {e}")
    return None


print(safe_divide(10, '2'))       # works fine → 5.0
print(safe_divide(10, 'abc'))     # ValueError
print(safe_divide(10, '0'))       # ZeroDivisionError

## 1.3 — `else` and `finally`

The full anatomy of exception handling:

```
try:
    <risky code>
except SomeError:
    <runs only if that error was raised>
else:
    <runs only if NO exception was raised>
finally:
    <ALWAYS runs — perfect for cleanup>
```

- **`else`** — use it for code that should run only on success (keeps the `try` block focused on the risky part).
- **`finally`** — use it to release resources (close files, database connections, etc.) regardless of what happened.

In [ ]:
def divide_with_full_handling(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  ❌ Division by zero!")
    else:
        # Only reached if try succeeded
        print(f"  ✅ Result = {result:.4f}")
    finally:
        # Always reached
        print("  🔚 Calculation attempt complete.\n")

divide_with_full_handling(10, 3)
divide_with_full_handling(10, 0)

## 1.4 — The EAFP Philosophy 🦅

> **"It is Easier to Ask for Forgiveness than Permission."**  
> — Grace Hopper (and a core Python idiom)

Python culture favours **EAFP** over its alternative **LBYL** (Look Before You Leap).

| Style | Approach | Python preference |
|---|---|---|
| **LBYL** | Check first, then act | ❌ Less Pythonic |
| **EAFP** | Just try it, catch errors if they happen | ✅ More Pythonic |

### Why prefer EAFP?

1. **Race conditions** — the world can change between your check and your action (e.g. a file could be deleted after you checked it exists).
2. **Cleaner code** — the "happy path" stays uncluttered by guard conditions.
3. **It matches Python internals** — Python itself uses this style everywhere.

Think of it like this: a good cook doesn't taste every ingredient before putting it in the pot — they cook, and *if* something is off, they fix it. They ask for forgiveness, not permission.

In [ ]:
# ──────────────────────────────────────────────────
# Example 1 — Accessing a dictionary key
# ──────────────────────────────────────────────────

user = {"name": "Alice", "age": 30}

# ❌ LBYL style — check before accessing
if "email" in user:
    print(user["email"])
else:
    print("LBYL: email not found")

# ✅ EAFP style — just try it
try:
    print(user["email"])
except KeyError:
    print("EAFP: email not found")

In [ ]:
# ──────────────────────────────────────────────────
# Example 2 — Type conversion
# ──────────────────────────────────────────────────

raw_inputs = ["42", "hello", "7", "3.14", ""]

# ✅ EAFP — try to convert, handle failure gracefully
numbers = []
for item in raw_inputs:
    try:
        numbers.append(int(item))
    except (ValueError, TypeError):
        print(f"  ⚠️  Could not convert '{item}' to int — skipping.")

print(f"\nValid numbers collected: {numbers}")

In [ ]:
# ──────────────────────────────────────────────────
# Example 3 — File handling (EAFP shines here)
# ──────────────────────────────────────────────────

# ❌ LBYL — check if file exists first (but the file could vanish after the check!)
import os
if os.path.exists("data.txt"):
    with open("data.txt") as f:
        print(f.read())

# ✅ EAFP — just open it; handle the error if it doesn't exist
try:
    with open("data.txt") as f:
        content = f.read()
        print(content)
except FileNotFoundError:
    print("EAFP: 'data.txt' was not found — creating it now.")
    with open("data.txt", "w") as f:
        f.write("Hello from EAFP!\n")
    print("File created successfully.")
finally:
    print("File operation complete.")

## 1.5 — Custom Exceptions

You can create your own exception types by inheriting from `Exception`.  
This makes your error handling expressive and self-documenting.

In [ ]:
# Define custom exception classes
class InsufficientFundsError(Exception):
    """Raised when a bank withdrawal exceeds the available balance."""
    def __init__(self, amount, balance):
        self.amount = amount
        self.balance = balance
        super().__init__(f"Cannot withdraw {amount:.2f} — balance is only {balance:.2f}")


class NegativeAmountError(Exception):
    """Raised when a negative amount is supplied."""
    pass


# A simple account that uses these custom errors
class BankAccount:
    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Deposit amount must be positive.")
        self.balance += amount
        print(f"  ✅ Deposited {amount:.2f}. New balance: {self.balance:.2f}")

    def withdraw(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Withdrawal amount must be positive.")
        if amount > self.balance:
            raise InsufficientFundsError(amount, self.balance)
        self.balance -= amount
        print(f"  ✅ Withdrew {amount:.2f}. New balance: {self.balance:.2f}")


# EAFP in action with custom exceptions
account = BankAccount("Alice", 1000.0)

for operation, amount in [("deposit", 500), ("withdraw", 200), ("withdraw", 5000), ("deposit", -50)]:
    print(f"\nAttempting {operation}({amount}):")
    try:
        if operation == "deposit":
            account.deposit(amount)
        else:
            account.withdraw(amount)
    except InsufficientFundsError as e:
        print(f"  ❌ InsufficientFundsError: {e}")
    except NegativeAmountError as e:
        print(f"  ❌ NegativeAmountError: {e}")

---
# Part 2 — OOP: Classes & Objects

## What is Object-Oriented Programming?

**Object-Oriented Programming (OOP)** is a way of organising code around *things* (objects) rather than *actions* (functions).

A **class** is a blueprint — like an architectural drawing.  
An **object** (instance) is what you build *from* that blueprint — like the actual house.

Every object has:
- **Attributes** — data it holds (e.g. a dog's name, breed, age)
- **Methods** — actions it can perform (e.g. `bark()`, `fetch()`)

OOP is built on **4 pillars**: Encapsulation, Inheritance, Polymorphism, Abstraction.

In [ ]:
# ──────────────────────────────────────────────────
# Defining a class and creating objects
# ──────────────────────────────────────────────────

class Dog:
    """
    Represents a dog.

    __init__ is the constructor — Python calls it automatically
    when you create a new Dog object.
    'self' always refers to the specific object being created.
    """
    # Class attribute — shared by ALL instances
    species = "Canis lupus familiaris"

    def __init__(self, name: str, breed: str, age: int):
        # Instance attributes — unique to each Dog object
        self.name = name
        self.breed = breed
        self.age = age

    def bark(self):
        """Instance method — called on a specific dog."""
        print(f"{self.name} says: Woof! 🐶")

    def description(self):
        return f"{self.name} is a {self.age}-year-old {self.breed}."


# Create two different Dog objects from the same blueprint
dog1 = Dog("Buddy", "Labrador", 3)
dog2 = Dog("Luna",  "Husky",    5)

dog1.bark()
dog2.bark()
print(dog1.description())
print(dog2.description())

# Class attribute is the same for both
print(f"\nBoth are of species: {dog1.species}")
print(f"Are they the same object? {dog1 is dog2}")

In [ ]:
# ──────────────────────────────────────────────────
# Practical example: a richer BankAccount class
# ──────────────────────────────────────────────────

class BankAccount:
    """A simple bank account that logs all transactions."""

    def __init__(self, owner: str, balance: float = 0.0):
        self.owner = owner
        self._balance = balance        # convention: _ means "treat as private"
        self._transactions = []        # history log

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self._balance += amount
        self._transactions.append(("deposit", amount))
        print(f"  + Deposited ${amount:,.2f}  |  Balance: ${self._balance:,.2f}")

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive.")
        if amount > self._balance:
            raise ValueError(f"Insufficient funds. Balance: ${self._balance:,.2f}")
        self._balance -= amount
        self._transactions.append(("withdrawal", amount))
        print(f"  - Withdrew  ${amount:,.2f}  |  Balance: ${self._balance:,.2f}")

    def get_balance(self) -> float:
        return self._balance

    def print_statement(self) -> None:
        print(f"\n{'='*40}")
        print(f"  Statement for: {self.owner}")
        print(f"{'='*40}")
        for txn_type, amount in self._transactions:
            sign = "+" if txn_type == "deposit" else "-"
            print(f"  {sign} ${amount:>10,.2f}  ({txn_type})")
        print(f"{'-'*40}")
        print(f"  Current balance: ${self._balance:,.2f}")
        print(f"{'='*40}")


acc = BankAccount("Alice", 1000.0)
acc.deposit(500)
acc.deposit(250)
acc.withdraw(100)

# EAFP: try a bad withdrawal
try:
    acc.withdraw(99999)
except ValueError as e:
    print(f"  ❌ {e}")

acc.print_statement()

---
# Part 3 — Inheritance

## What is Inheritance?

**Inheritance** lets a new class *reuse* the code from an existing class, then extend or override it.

- The original class is the **parent** (or base / superclass).
- The new class is the **child** (or derived / subclass).
- `super()` gives you access to the parent's methods.

Think of it biologically: a child inherits traits from a parent but can also have their own unique characteristics.

In [ ]:
# ──────────────────────────────────────────────────
# Single Inheritance
# ──────────────────────────────────────────────────

class Vehicle:
    """Base class for all vehicles."""

    def __init__(self, make: str, model: str, year: int):
        self.make = make
        self.model = model
        self.year = year

    def describe(self) -> str:
        return f"{self.year} {self.make} {self.model}"

    def start(self) -> None:
        print(f"  {self.describe()} is starting... 🚗")


class ElectricCar(Vehicle):
    """A Vehicle that runs on electricity — extends the base class."""

    def __init__(self, make: str, model: str, year: int, battery_kWh: int):
        super().__init__(make, model, year)   # reuse parent __init__
        self.battery_kWh = battery_kWh

    def describe(self) -> str:
        # Override parent's describe() to add battery info
        base = super().describe()
        return f"{base} (Electric, {self.battery_kWh} kWh)"

    def charge(self) -> None:
        print(f"  ⚡ {self.describe()} is charging...")


# Regular car uses Vehicle directly
car    = Vehicle("Toyota", "Camry", 2022)
tesla  = ElectricCar("Tesla", "Model 3", 2024, 82)

car.start()
tesla.start()    # inherited from Vehicle
tesla.charge()   # new method in ElectricCar

print(f"\nIs tesla a Vehicle? {isinstance(tesla, Vehicle)}")
print(f"Is car an ElectricCar? {isinstance(car, ElectricCar)}")

In [ ]:
# ──────────────────────────────────────────────────
# Multiple Inheritance
# When a class inherits from more than one parent.
# ──────────────────────────────────────────────────

class Flyable:
    """Mixin: gives a class the ability to fly."""
    def fly(self):
        print(f"  🛫 {self.__class__.__name__} is flying!")


class Swimmable:
    """Mixin: gives a class the ability to swim."""
    def swim(self):
        print(f"  🌊 {self.__class__.__name__} is swimming!")


class Animal:
    def __init__(self, name: str):
        self.name = name

    def breathe(self):
        print(f"  💨 {self.name} is breathing.")


class Duck(Animal, Flyable, Swimmable):
    """A duck can do everything — inherits from three classes."""
    def quack(self):
        print(f"  🦆 {self.name} says: Quack!")


donald = Duck("Donald")
donald.breathe()   # from Animal
donald.fly()       # from Flyable
donald.swim()      # from Swimmable
donald.quack()     # Duck's own method

# Python's Method Resolution Order (MRO) shows the lookup chain
print(f"\nMRO: {[c.__name__ for c in Duck.__mro__]}")

---
# Part 4 — Polymorphism

## What is Polymorphism?

**Polymorphism** means "many forms". In OOP it means: the **same method name** produces **different behaviour** depending on the object you call it on.

It's the magic that lets you write one function that works on any object that has a certain method — without caring exactly what class the object is.

Real-world analogy: a "speak" command means something different if you give it to a dog vs. a parrot vs. a human — but you issue the same command.

In [ ]:
# ──────────────────────────────────────────────────
# Method Overriding — child redefines a parent method
# ──────────────────────────────────────────────────

class Shape:
    """Base class. area() and perimeter() MUST be overridden."""
    def area(self) -> float:
        raise NotImplementedError("Subclasses must implement area()")

    def perimeter(self) -> float:
        raise NotImplementedError("Subclasses must implement perimeter()")

    def describe(self) -> None:
        # This method is polymorphic — it calls area() and perimeter()
        # which will resolve to the correct subclass version at runtime
        print(f"  {self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}")


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


class Circle(Shape):
    PI = 3.14159265

    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return self.PI * self.radius ** 2

    def perimeter(self) -> float:
        return 2 * self.PI * self.radius


class Triangle(Shape):
    def __init__(self, a: float, b: float, c: float):
        self.a, self.b, self.c = a, b, c

    def perimeter(self) -> float:
        return self.a + self.b + self.c

    def area(self) -> float:
        # Heron's formula
        s = self.perimeter() / 2
        return (s * (s-self.a) * (s-self.b) * (s-self.c)) ** 0.5


# One function — works for ANY shape
def print_shape_info(shape: Shape) -> None:
    shape.describe()


shapes = [
    Rectangle(4, 6),
    Circle(5),
    Triangle(3, 4, 5),
]

print("Shape information:")
for s in shapes:
    print_shape_info(s)   # same call — different result each time!

---
# Part 5 — Encapsulation

## What is Encapsulation?

**Encapsulation** bundles data and the methods that operate on it inside a class, and **controls** how that data is accessed from the outside.

Python uses naming conventions (not hard enforcement) for access control:

| Prefix | Convention | Meaning |
|---|---|---|
| `name` | No prefix | **Public** — anyone can access |
| `_name` | Single underscore | **Protected** — "don't touch from outside, please" |
| `__name` | Double underscore | **Private** — Python applies *name mangling* to hide it |

**Why restrict access?**  
To protect your object's internal state from being accidentally broken by outside code — and to let you change the implementation later without breaking anything that uses your class.

In [ ]:
# ──────────────────────────────────────────────────
# Demonstrating access levels
# ──────────────────────────────────────────────────

class Person:
    def __init__(self, name: str, age: int, salary: float):
        self.name     = name     # public   — free to read and write
        self._age     = age      # protected — read is okay, be careful with writes
        self.__salary = salary   # private  — only accessible via class methods

    # Getter: controlled read access
    def get_salary(self) -> float:
        return self.__salary

    # Setter: controlled write access — we can add validation here
    def set_salary(self, new_salary: float) -> None:
        if new_salary < 0:
            raise ValueError("Salary cannot be negative.")
        print(f"  Salary updated: {self.__salary:,.2f} → {new_salary:,.2f}")
        self.__salary = new_salary

    def __repr__(self):
        return f"Person(name={self.name!r}, age={self._age})"


p = Person("Bob", 28, 55000.0)

# ✅ Public — works fine
print(p.name)

# ✅ Protected — technically accessible, but you should avoid it
print(p._age)

# ✅ Private — use the getter
print(p.get_salary())

# ✅ Use the setter (which validates)
p.set_salary(60000)

# ❌ Trying to set a negative salary is caught
try:
    p.set_salary(-500)
except ValueError as e:
    print(f"  ❌ {e}")

# ❌ Directly accessing __salary fails (name mangling)
try:
    print(p.__salary)
except AttributeError as e:
    print(f"  ❌ {e}")

# The mangled name still technically works (but never do this!)
print(f"  (Mangled name access: {p._Person__salary})")

In [ ]:
# ──────────────────────────────────────────────────
# Pythonic encapsulation with @property
# The cleanest way to add getters and setters
# ──────────────────────────────────────────────────

class Temperature:
    """Stores temperature in Celsius internally, but exposes both scales."""

    def __init__(self, celsius: float):
        self.celsius = celsius   # calls the setter below

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError(f"{value}°C is below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self) -> float:
        """Computed property — derived from celsius."""
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value: float) -> None:
        self.celsius = (value - 32) * 5/9

    def __repr__(self):
        return f"Temperature({self.celsius:.1f}°C / {self.fahrenheit:.1f}°F)"


t = Temperature(100)   # boiling point
print(t)

t.fahrenheit = 32      # set via Fahrenheit
print(t)               # automatically updated in Celsius too

# EAFP — ask for forgiveness when setting an impossible temperature
try:
    t.celsius = -300
except ValueError as e:
    print(f"❌ {e}")

---
# Part 6 — Abstraction

## What is Abstraction?

**Abstraction** hides complex internal details and exposes only what the user of a class needs to know.

The key tool is the **Abstract Base Class (ABC)**:
- You declare methods as `@abstractmethod` — these are *contracts*.
- Any subclass **must** implement them, or Python raises a `TypeError`.
- You cannot instantiate the abstract class itself.

Think of an ABC as an interface or a job description: "Any class that claims to be a `Shape` must be able to compute its area and perimeter."

In [ ]:
from abc import ABC, abstractmethod


class PaymentProcessor(ABC):
    """
    Abstract base class for payment processors.
    Defines the CONTRACT that every payment method must fulfil.
    """

    @abstractmethod
    def pay(self, amount: float) -> None:
        """Process a payment of the given amount."""
        pass

    @abstractmethod
    def refund(self, amount: float) -> None:
        """Refund a payment."""
        pass

    # Non-abstract method — shared behaviour for all processors
    def validate_amount(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(f"Invalid amount: {amount}. Must be > 0.")


class CreditCardProcessor(PaymentProcessor):
    def __init__(self, card_number: str):
        self.card_number = f"****{card_number[-4:]}"  # mask card number

    def pay(self, amount: float) -> None:
        self.validate_amount(amount)
        print(f"  💳 Credit card {self.card_number}: charged ${amount:.2f}")

    def refund(self, amount: float) -> None:
        self.validate_amount(amount)
        print(f"  💳 Credit card {self.card_number}: refunded ${amount:.2f}")


class PayPalProcessor(PaymentProcessor):
    def __init__(self, email: str):
        self.email = email

    def pay(self, amount: float) -> None:
        self.validate_amount(amount)
        print(f"  🅿️  PayPal ({self.email}): sent ${amount:.2f}")

    def refund(self, amount: float) -> None:
        self.validate_amount(amount)
        print(f"  🅿️  PayPal ({self.email}): refunded ${amount:.2f}")


# ─────────────────────────────────────
# The caller doesn't need to know WHICH processor is being used.
# This is abstraction + polymorphism working together.
# ─────────────────────────────────────
def checkout(processor: PaymentProcessor, amount: float) -> None:
    try:
        processor.pay(amount)
    except ValueError as e:
        print(f"  ❌ Payment failed: {e}")


# Prove you cannot instantiate an abstract class
try:
    p = PaymentProcessor()
except TypeError as e:
    print(f"❌ Cannot instantiate ABC: {e}\n")

# Use concrete implementations
processors = [
    CreditCardProcessor("1234567890123456"),
    PayPalProcessor("alice@example.com"),
]

for proc in processors:
    checkout(proc, 99.99)

---
# Part 7 — Magic Methods & Operator Overloading

## What are Magic Methods?

**Magic methods** (also called *dunder methods* from **d**ouble **under**score) let you define how your objects behave with Python's built-in syntax and functions.

| Method | Triggered by |
|---|---|
| `__init__` | `MyClass()` — object creation |
| `__str__` | `print(obj)` or `str(obj)` |
| `__repr__` | `repr(obj)` — developer-friendly string |
| `__len__` | `len(obj)` |
| `__add__` | `obj1 + obj2` |
| `__eq__` | `obj1 == obj2` |
| `__lt__` | `obj1 < obj2` |
| `__contains__` | `item in obj` |
| `__getitem__` | `obj[key]` |
| `__iter__` | `for item in obj` |

In [ ]:
# ──────────────────────────────────────────────────
# __str__ vs __repr__
# ──────────────────────────────────────────────────

class Book:
    def __init__(self, title: str, author: str, pages: int):
        self.title = title
        self.author = author
        self.pages = pages

    def __str__(self) -> str:
        """Human-readable — for print() and str()."""
        return f"'{self.title}' by {self.author}"

    def __repr__(self) -> str:
        """Developer-readable — should look like a constructor call."""
        return f"Book(title={self.title!r}, author={self.author!r}, pages={self.pages})"

    def __len__(self) -> int:
        """len(book) returns the page count."""
        return self.pages


book = Book("The Pragmatic Programmer", "David Thomas", 352)

print(str(book))    # uses __str__
print(repr(book))   # uses __repr__
print(len(book))    # uses __len__

In [ ]:
# ──────────────────────────────────────────────────
# Operator Overloading — Vector math
# ──────────────────────────────────────────────────

import math

class Vector:
    """
    A 2D mathematical vector.
    Supports +, -, *, /, ==, <, abs(), len(), and print().
    """

    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    # String representations
    def __str__(self)  -> str:  return f"Vector({self.x}, {self.y})"
    def __repr__(self) -> str:  return f"Vector({self.x!r}, {self.y!r})"

    # Arithmetic
    def __add__(self, other: "Vector") -> "Vector":
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Vector") -> "Vector":
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar) -> "Vector":
        """Scalar multiplication: Vector * 3"""
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar) -> "Vector":
        """Reverse scalar multiplication: 3 * Vector"""
        return self.__mul__(scalar)

    def __truediv__(self, scalar) -> "Vector":
        return Vector(self.x / scalar, self.y / scalar)

    def __neg__(self) -> "Vector":
        """Unary negation: -v"""
        return Vector(-self.x, -self.y)

    # Comparison
    def __eq__(self, other) -> bool:
        return self.x == other.x and self.y == other.y

    def __lt__(self, other) -> bool:
        """Compare by magnitude."""
        return abs(self) < abs(other)

    # Built-in functions
    def __abs__(self) -> float:
        """abs(v) returns the vector's magnitude."""
        return math.sqrt(self.x**2 + self.y**2)

    def dot(self, other: "Vector") -> float:
        """Dot product (not a magic method, but useful)."""
        return self.x * other.x + self.y * other.y


v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(f"v1          = {v1}")
print(f"v2          = {v2}")
print(f"v1 + v2     = {v1 + v2}")
print(f"v1 - v2     = {v1 - v2}")
print(f"v1 * 3      = {v1 * 3}")
print(f"2 * v2      = {2 * v2}")
print(f"-v1         = {-v1}")
print(f"abs(v1)     = {abs(v1):.4f}   (magnitude)")
print(f"v1 == v1    = {v1 == v1}")
print(f"v1 == v2    = {v1 == v2}")
print(f"v2 < v1     = {v2 < v1}")
print(f"v1 · v2     = {v1.dot(v2)}    (dot product)")

In [ ]:
# ──────────────────────────────────────────────────
# A custom collection — __len__, __getitem__, __contains__, __iter__
# ──────────────────────────────────────────────────

class Playlist:
    """A simple music playlist that behaves like a Python sequence."""

    def __init__(self, name: str):
        self.name = name
        self._songs = []

    def add(self, song: str) -> None:
        self._songs.append(song)

    def __len__(self)  -> int:    return len(self._songs)
    def __str__(self)  -> str:    return f"Playlist '{self.name}' ({len(self)} songs)"
    def __repr__(self) -> str:    return f"Playlist(name={self.name!r}, songs={self._songs!r})"

    def __getitem__(self, index): return self._songs[index]
    def __contains__(self, song): return song in self._songs
    def __iter__(self):           return iter(self._songs)

    def __add__(self, other: "Playlist") -> "Playlist":
        """Merge two playlists with +."""
        merged = Playlist(f"{self.name} + {other.name}")
        merged._songs = self._songs + other._songs
        return merged


p1 = Playlist("Morning Vibes")
for song in ["Sunrise", "Coffee Time", "Wake Up"]:
    p1.add(song)

p2 = Playlist("Evening Chill")
for song in ["Sunset", "Mellow Tones"]:
    p2.add(song)

print(p1)
print(f"First song  : {p1[0]}")
print(f"Last song   : {p1[-1]}")
print(f"'Wake Up' in p1? {'Wake Up' in p1}")
print(f"'Sunset' in p1?  {'Sunset' in p1}")

# Merge with +
full_day = p1 + p2
print(f"\n{full_day}")
print("Tracklist:")
for i, song in enumerate(full_day, 1):
    print(f"  {i}. {song}")

---
# 🏁 Summary & Cheat Sheet

## Exception Handling

```python
try:
    risky_code()
except SpecificError as e:   # most specific first
    handle(e)
except Exception as e:        # catch-all last
    handle(e)
else:
    success_code()            # only if try succeeded
finally:
    cleanup()                 # always runs
```

**EAFP** (Easier to Ask Forgiveness than Permission): just try it, catch errors → more Pythonic than checking everything upfront.

---

## OOP: The Four Pillars

| Pillar | What it does | Key syntax |
|---|---|---|
| **Encapsulation** | Bundles data + methods; controls access | `_protected`, `__private`, `@property` |
| **Inheritance** | Child class reuses parent's code | `class Child(Parent)`, `super()` |
| **Polymorphism** | Same method name, different behaviour | Method overriding |
| **Abstraction** | Hide details, expose a clean interface | `ABC`, `@abstractmethod` |

---

## Key Magic Methods

```python
__init__      # constructor
__str__       # print(obj)
__repr__      # repr(obj)
__len__       # len(obj)
__add__       # obj + other
__eq__        # obj == other
__lt__        # obj < other
__contains__  # item in obj
__iter__      # for item in obj
__getitem__   # obj[key]
```